# 05.5 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/05-powerful-models/05-svm/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

from sklearn.datasets import make_blobs, make_moons
from sklearn.svm import SVC, SVR

X_blob, y_blob = make_blobs(n_samples=40, centers=2, random_state=6)

def plot_svm(model, X, y, ax, title):
    """Scatter the points, shade the decision function, draw boundary, margins and support vectors."""
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200),
                         np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 200))
    Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z > 0, alpha=0.15, cmap="coolwarm")
    ax.contour(xx, yy, Z, levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"], linewidths=1)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=25, edgecolors="k")
    sv = model.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=120, facecolors="none", edgecolors="k", linewidths=1.5)
    ax.set_title(title, fontsize=9); ax.set_xticks([]); ax.set_yticks([])

## Exercise 1

$\lVert w \rVert = \sqrt{3^2 + 4^2} = \sqrt{25} = 5$, so the margin is $2 / 5 = 0.4$.

In [2]:
print(2 / np.linalg.norm([3, 4]))

0.4


## Exercise 2

In [3]:
base = SVC(kernel="linear", C=1).fit(X_blob, y_blob)
sv_idx = set(base.support_)
non_sv = next(i for i in range(len(X_blob)) if i not in sv_idx)
one_sv = base.support_[0]

for label, drop in [("drop a non-support point", non_sv), ("drop a support vector", one_sv)]:
    keep = np.arange(len(X_blob)) != drop
    m = SVC(kernel="linear", C=1).fit(X_blob[keep], y_blob[keep])
    print(f"{label:<26} coef {m.coef_[0].round(3)}  intercept {m.intercept_[0]:.3f}")
print(f"{'original':<26} coef {base.coef_[0].round(3)}  intercept {base.intercept_[0]:.3f}")

drop a non-support point   coef [-0.254 -0.838]  intercept -3.211
drop a support vector      coef [-0.161 -0.773]  intercept -3.316
original                   coef [-0.254 -0.838]  intercept -3.211


Dropping a non-support point leaves the boundary exactly where it was. Dropping a support vector moves it, because the corridor was resting on that point.

## Exercise 3

In [4]:
from sklearn.model_selection import cross_val_score
X_moon, y_moon = make_moons(n_samples=300, noise=0.25, random_state=0)
grid = pd.DataFrame(index=[0.1, 1, 10, 100], columns=[0.1, 1, 10, 100], dtype=float)
for C in grid.index:
    for g in grid.columns:
        grid.loc[C, g] = cross_val_score(SVC(C=C, gamma=g), X_moon, y_moon, cv=5).mean()
grid.index.name, grid.columns.name = "C", "gamma"
print("best:", grid.stack().idxmax(), f"{grid.stack().max():.3f}")
grid.round(3)

best: (np.float64(10.0), np.float64(1.0)) 0.963


gamma,0.1,1.0,10.0,100.0
C,,,,
0.1,0.810,0.907,0.953,0.880
1.0,0.837,0.957,0.960,0.927
10.0,0.870,0.963,0.953,0.917
100.0,0.943,0.950,0.923,0.917


The best cell is `C = 10`, `gamma = 1` (0.963), in the middle of the grid. Low `C` with low `gamma` underfits badly (0.810: an almost straight, very tolerant boundary). `gamma = 100` overfits at every `C` (0.88–0.93). And notice that a weak setting of one knob can be partly rescued by the other: with `gamma = 0.1`, raising `C` to 100 lifts accuracy from 0.81 to 0.94. This is why `C` and `gamma` must be tuned **together**, which is exactly what grid search (05.6) automates.

## Exercise 4

In [5]:
import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail
from sklearn.model_selection import train_test_split

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]
cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)
cust = cust.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}, "
      f"train {len(X_train):,} / test {len(X_test):,}")

3,317 customers, churn rate 43.6%, train 2,487 / test 830


In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

def cv_auc(model):
    """Mean 5-fold cross-validated AUC on the training customers."""
    return cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc").mean()

def test_auc(model):
    """Fit on all training customers, score AUC on the held-out test customers."""
    model.fit(X_train, y_train)
    return roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

print(f"Benchmark logistic regression: CV AUC {cv_auc(logreg):.3f}, test AUC {test_auc(logreg):.3f}")

Benchmark logistic regression: CV AUC 0.740, test AUC 0.741


In [7]:
for C in [0.1, 1, 10]:
    m = make_pipeline(StandardScaler(), SVC(C=C, probability=True, random_state=42))
    print(f"SVM C={C:>4}: CV AUC {cv_auc(m):.3f}")

SVM C= 0.1: CV AUC 0.731


SVM C=   1: CV AUC 0.731


SVM C=  10: CV AUC 0.734


Compare with 0.740: the tuned SVM lands in the same region as the other models on this table, without beating logistic regression, and takes noticeably longer to cross-validate. For ShopWave churn, the SVM adds cost without adding accuracy.